# CCM-v2: Multi-cell Cellular Consensus vs SmolLM2-360M-Instruct — chat and fidelity test

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vtavakkoli/TinyCeNN-LM/blob/research/competitive-cenn-feasibility/notebooks/CCM_v2_SmolLM2_360M_Consensus_Chat_Colab.ipynb)

**Objective:** Evaluate whether *a single* native first-attention mixer (`model.layers[0].self_attn`) can be replaced by an independently trained **finite cellular memory** with multi-cell mass-conserving aggregation. Compare actual SmolLM2-360M-Instruct language-model quality, native-vs-cellular **chat answers**, and **full-model GPU prefill** latency. Auto ZIP download at the end.

**Scientific honesty:** Although the implementation performs local cellular merges and never calls an original attention kernel, the log-mass aggregation is **mathematically equivalent to softmax attention over the resident bounded cells**. This is a feasibility/upper-bound control, **not proven novel attention-free computation**. The remainder of the model still uses native attention; this is one-layer substitution only. Cached decoding isn't supported; all chats use fair full-prefix greedy decoding. **No claim of parity or speedup is made before the GPU run.**

**Why this notebook:** Your earlier diagnostic found top-16 attention mass only ≈50.6% at 128 tokens and ≈35.6% at 256; a hard winner is not a sufficient representation. CCM-v2 retains contributions from multiple cells and learns independent query/key projections. Its memory window remains a hard limit.


In [ ]:
#@title 1. Install reproducible CPU-safe requirements, clone research branch (no file upload)
import os, sys, subprocess, json, pathlib, torch
print('Installed torch:', torch.__version__)
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '(no GPU)')
assert torch.cuda.is_available(), 'Select Runtime → Change runtime type → T4 GPU / L4 GPU first'
subprocess.check_call([sys.executable,'-m','pip','-q','install','transformers==4.57.6',
                       'huggingface_hub>=0.34,<1','datasets>=3,<5','pytest','pandas','matplotlib','safetensors'])
repo=pathlib.Path('/content/TinyCeNN-LM')
if not repo.exists():
    subprocess.check_call(['git','clone','--depth','1','--branch','research/competitive-cenn-feasibility',
                           'https://github.com/vtavakkoli/TinyCeNN-LM.git',str(repo)])
else:
    subprocess.check_call(['git','-C',str(repo),'fetch','origin','research/competitive-cenn-feasibility','--depth','1'])
    subprocess.check_call(['git','-C',str(repo),'checkout','-B','research/competitive-cenn-feasibility','FETCH_HEAD'])
subprocess.check_call([sys.executable,'-m','pip','-q','install','-e',str(repo),'--no-deps'])
sys.path.insert(0,str(repo))
print('Git SHA:',subprocess.check_output(['git','-C',str(repo),'rev-parse','HEAD'],text=True).strip())

In [ ]:
#@title 2. CPU mathematical and adapter contract tests
subprocess.check_call([sys.executable,'-m','pytest','-q',
                       str(repo/'tests/test_competitive_consensus_v2.py'),
                       str(repo/'tests/test_ccm_consensus_smol360.py')])

In [ ]:
#@title 3. Experiment controls (GPU)
PROFILE = 'quick'   #@param ['smoke','quick','research']
SLOTS = 128         #@param {type:'integer'}
TAG_WIDTH = 64      #@param {type:'integer'}
CELL_WIDTH = 256    #@param {type:'integer'}
SCORE_SCALE = 2.0   #@param {type:'number'}
SEED = 2031         #@param {type:'integer'}
CHAT_MAX_NEW_TOKENS = 48  #@param {type:'integer'}
# Recommended: 'smoke' to verify execution; 'quick' for research screening,
# 'research' for longer training. SLOTS must be a power of two.
# Increase slots to 256 for the 256-token context only if memory allows.
OUTPUT_DIR = pathlib.Path('/content/CCM_v2_360M_Consensus_Chat_Results')
OUTPUT_DIR.mkdir(exist_ok=True,parents=True)
CHAT_PROMPTS = [
    {'id':'fact','user':'What is the capital of Austria? Answer in one sentence.'},
    {'id':'arithmetic','user':'What is 17 plus 25? Reply with the number only.'},
    {'id':'explanation','user':'Explain the difference between RAM and SSD in two short sentences.'},
    {'id':'format','user':'Respond with exactly three words about the Moon.'},
    {'id':'recall','user':'Remember this code: AURORA-731. What was the code I just gave you?'},
]
(OUTPUT_DIR/'chat_prompts.json').write_text(json.dumps(CHAT_PROMPTS,indent=2))
print('Chat cases:',len(CHAT_PROMPTS),'results:',OUTPUT_DIR)

In [ ]:
#@title 4. Train CCM-v2; DEV selection → sealed TEST → CUDA timing → chat comparison
from scripts.benchmark_ccm_consensus_smol360 import main
import time
args=['--profile', PROFILE, '--output-dir',str(OUTPUT_DIR),'--slots',str(SLOTS),
      '--tag-width',str(TAG_WIDTH),'--cell-width',str(CELL_WIDTH),
      '--score-scale',str(SCORE_SCALE),'--seed',str(SEED),
      '--chat-new-tokens',str(CHAT_MAX_NEW_TOKENS),
      '--chat-prompts-json',str(OUTPUT_DIR/'chat_prompts.json')]
report, session = main(args, return_runtime=True)
print('Validation ΔNLL:',report['dev']['delta_nll'])
print('DEV real gold-token accuracy: native',report['dev']['teacher_gold_accuracy'],
      'candidate',report['dev']['student_gold_accuracy'])
print('Teacher top-1 agreement (diagnostic ONLY):',report['dev']['teacher_agreement'])
print('Sealed TEST ΔNLL:',report['test']['delta_nll'])
print('TEST real gold-token accuracy: native',report['test']['teacher_gold_accuracy'],
      'candidate',report['test']['student_gold_accuracy'])
print('Full-model speedup:',report['paired_latency'].get('speedup'))
print('ACCEPTED:',report['accepted'])

In [ ]:
#@title 5. Plot learning curves and show a machine-readable scorecard
import pandas as pd, matplotlib.pyplot as plt
from IPython.display import display
frame=pd.read_csv(OUTPUT_DIR/'dev_history.csv')
display(frame[['step','phase','cosine','delta_nll','student_gold_accuracy','teacher_gold_accuracy','teacher_agreement']])
for field in ['student_gold_accuracy','delta_nll','cosine']:
    plt.figure(figsize=(7,3))
    plt.plot(frame['step'],frame[field],marker='o')
    plt.xlabel('Optimization step');plt.ylabel(field);plt.grid(True,alpha=0.2)
    plt.title('CCM-v2 DEV '+field)
    plt.tight_layout();plt.savefig(OUTPUT_DIR/('dev_'+field+'.png'),dpi=140);plt.show()
score=pd.DataFrame([
 {'metric':'ΔNLL','DEV':report['dev']['delta_nll'],'TEST':report['test']['delta_nll']},
 {'metric':'Gold-token accuracy (native)','DEV':report['dev']['teacher_gold_accuracy'],'TEST':report['test']['teacher_gold_accuracy']},
 {'metric':'Gold-token accuracy (CCM)','DEV':report['dev']['student_gold_accuracy'],'TEST':report['test']['student_gold_accuracy']},
 {'metric':'Native NLL','DEV':report['dev']['teacher_nll'],'TEST':report['test']['teacher_nll']},
 {'metric':'CCM NLL','DEV':report['dev']['student_nll'],'TEST':report['test']['student_nll']},
])
display(score)
if not report['accepted']:
    print('NEGATIVE RESULT: Do NOT replace the deployed native first attention module.')

In [ ]:
#@title 6. Chat comparison table — identical prompts and greedy no-cache generation
chat=pd.read_json(OUTPUT_DIR/'chat_comparison.json')
if len(chat):
    simple=pd.DataFrame([{
      'case':c['case'],'prompt':c['prompt'],
      'original answer':c['teacher']['response'],
      'CCM answer':c['ccm']['response'],
      'exact match':c['exact_response_match']} for c in json.loads((OUTPUT_DIR/'chat_comparison.json').read_text())])
    display(simple)
else:print('No chat responses generated; check --skip-chat setting.')
print('Exact text agreement is diagnostic only, not a factual correctness score.')

In [ ]:
#@title 7. Optional custom chat prompt — edit text then rerun THIS cell
CUSTOM_PROMPT = ''  #@param {type:'string'}
if CUSTOM_PROMPT.strip():
    from scripts.benchmark_ccm_consensus_smol360 import chat_comparison
    custom=chat_comparison(session['model'],session['native'],session['adapter'],
         session['tokenizer'],[{'id':'custom','user':CUSTOM_PROMPT}],max_new_tokens=CHAT_MAX_NEW_TOKENS)
    (OUTPUT_DIR/'custom_chat.json').write_text(json.dumps(custom,indent=2))
else:
    print('Set CUSTOM_PROMPT to your question and rerun this cell to compare the original and CCM replies.')

## 8 · Ground-truth real fast evaluation

Use labeled ARC-Easy validation answer keys plus five explicit chat correctness rules. Teacher agreement and local cosine remain useful diagnostics but are not acceptance gates. The official WikiText DEV/test gold-token NLL and accuracy remain primary. This notebook does not implement cached decoding or demonstrate a fully standalone LM. No benchmark or teacher outputs are used for checkpoint selection after the sealed test.

In [ ]:
#@title 8. Real fast evaluation: ARC-Easy answer labels, rule-scored chats, and gold NLL
# No teacher response match is used as ground truth.
from scripts.real_fast_eval import arc_easy_eval,real_chat_eval,CHAT_CASES
from scripts.benchmark_ccm_consensus_smol360 import attention_slot
from datasets import load_dataset as load_eval_data
ARC_EXAMPLES=12 #@param {type:'integer'}
REAL_CHAT_TOKENS=48 #@param {type:'integer'}
if ARC_EXAMPLES<1:raise ValueError('ARC_EXAMPLES must be > 0')
arc_data=load_eval_data('allenai/ai2_arc','ARC-Easy',split='validation')
base_model=session['model']; native=session['native']; adapter=session['adapter']
tokenizer=session['tokenizer']
with attention_slot(base_model,native):
    real_arc_native=arc_easy_eval(base_model,tokenizer,dataset=arc_data,limit=ARC_EXAMPLES,seed=SEED)
with attention_slot(base_model,adapter):
    real_arc_ccm=arc_easy_eval(base_model,tokenizer,dataset=arc_data,limit=ARC_EXAMPLES,seed=SEED)
assert real_arc_native['evaluated']==real_arc_ccm['evaluated']>0
assert [x['id'] for x in real_arc_native['rows']]==[x['id'] for x in real_arc_ccm['rows']]
with attention_slot(base_model,native):
    real_chat_native=real_chat_eval(base_model,tokenizer,cases=CHAT_CASES,max_new_tokens=REAL_CHAT_TOKENS)
with attention_slot(base_model,adapter):
    real_chat_ccm=real_chat_eval(base_model,tokenizer,cases=CHAT_CASES,max_new_tokens=REAL_CHAT_TOKENS)
real_fast_report={
 'benchmark':'ARC-Easy validation, label log-likelihood (NOT teacher agreement)',
 'arc_native':real_arc_native,'arc_ccm':real_arc_ccm,
 'objective_chat_native':real_chat_native,'objective_chat_ccm':real_chat_ccm,
 'real_wikitext_dev':report['dev'],'real_wikitext_test':report['test'],
 'cuda_prefill':report['paired_latency'],
 'teacher_agreement_is_only_diagnostic':True,
 'warning':'Small ARC-Easy subset and five transparent chat rules; not an official benchmark. No cache.'}
(OUTPUT_DIR/'real_fast_eval.json').write_text(json.dumps(real_fast_report,indent=2))
full_report_path=OUTPUT_DIR/'final_report.json'
full_report=json.loads(full_report_path.read_text())
full_report['real_fast_eval_summary']={
 'arc_native_accuracy':real_arc_native['accuracy'],
 'arc_ccm_accuracy':real_arc_ccm['accuracy'],
 'chat_native_rule_pass_rate':real_chat_native['accuracy'],
 'chat_ccm_rule_pass_rate':real_chat_ccm['accuracy'],
 'wikitext_test_native_gold_accuracy':report['test']['teacher_gold_accuracy'],
 'wikitext_test_ccm_gold_accuracy':report['test']['student_gold_accuracy']}
full_report_path.write_text(json.dumps(full_report,indent=2))
print('Ground-truth ARC-Easy native/CCM:',real_arc_native['accuracy'],real_arc_ccm['accuracy'])
print('Objective chat correctness native/CCM:',real_chat_native['accuracy'],real_chat_ccm['accuracy'])
print('Sealed WikiText gold accuracy native/CCM:',
      report['test']['teacher_gold_accuracy'],report['test']['student_gold_accuracy'])
print('Sealed WikiText NLL native/CCM:',report['test']['teacher_nll'],report['test']['student_nll'])
print('Whole-model GPU prefill speedup:',report['paired_latency'].get('speedup'))
display(pd.DataFrame([{'task':a['id'],'native_correct':a['pass'],
 'CCM_correct':b['pass'],'native_response':a['response'],
 'CCM_response':b['response']} for a,b in zip(
 real_chat_native['rows'],real_chat_ccm['rows'])]))


In [ ]:
#@title 9. Package every result, training log, checkpoint and chat transcript; automatically download ZIP
import shutil
archive=shutil.make_archive(str(OUTPUT_DIR),'zip',root_dir=OUTPUT_DIR)
print('Generated:',archive)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print('Download the generated ZIP manually outside Google Colab')

## Interpreting the real gold-target outcome

- **Pass:** measured DEV/test ΔNLL ≤ 0.02, real next-token accuracy drop ≤ 2 percentage points, and full-model GPU speedup greater than 1.0×. These are screening acceptance gates, not evidence of scientific novelty.
- **Fail:** keep the native attention layer. The experiment saves its best **experimental** CCM checkpoint for diagnosis but never publishes a replacement model.
- **Chat tests:** native and candidate outputs are generated using the same prompt template, deterministic greedy choice and **full-prefix recomputation** without caching. They are qualitative sanity checks; a passing chat example does not imply full language-model parity.
- **Scope:** only `model.model.layers[0].self_attn` is replaced. The other attention layers are still native. CCM's mass consensus is mathematically softmax attention on a bounded memory tape, so the architecture is a stronger **upper-bound/feasibility control**, not proof of a novel attention-free algorithm.
- **Speed:** vectorized cellular prefill creates temporary `O(T × slots)` neighborhoods and is not a fused Triton kernel. It may be slower than native SDPA. Cached token-by-token generation and fully stand-alone model deployment require new implementations.
